In [1]:
# Cell 1 - Import libraries
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [2]:
# Cell 2 - Create Bollywood movie dataset
# We create our own dataset since no free Bollywood hit/flop dataset exists!
# This is called "synthetic dataset" — real ML technique used in industry!

data = {
    'budget': [150, 200, 50, 300, 25, 180, 100, 400, 30, 250,
               120, 80, 350, 45, 190, 75, 220, 60, 170, 90,
               500, 15, 280, 140, 70, 320, 55, 210, 95, 160,
               130, 85, 270, 40, 195, 65, 240, 110, 175, 35,
               450, 20, 290, 145, 72, 330, 58, 215, 98, 165],

    'genre': ['Action', 'Comedy', 'Horror', 'Action', 'Romance',
              'Thriller', 'Drama', 'Action', 'Comedy', 'Action',
              'Romance', 'Horror', 'Action', 'Drama', 'Comedy',
              'Thriller', 'Action', 'Romance', 'Drama', 'Horror',
              'Action', 'Comedy', 'Action', 'Romance', 'Horror',
              'Action', 'Drama', 'Comedy', 'Thriller', 'Action',
              'Romance', 'Horror', 'Action', 'Drama', 'Comedy',
              'Thriller', 'Action', 'Romance', 'Drama', 'Horror',
              'Action', 'Comedy', 'Action', 'Romance', 'Horror',
              'Action', 'Drama', 'Comedy', 'Thriller', 'Action'],

    'cast_popularity': [9, 7, 5, 10, 4, 8, 6, 9, 5, 8,
                        6, 4, 9, 5, 7, 6, 8, 5, 7, 4,
                        10, 3, 9, 7, 5, 9, 4, 8, 6, 8,
                        7, 5, 9, 4, 7, 6, 8, 6, 7, 3,
                        10, 4, 9, 7, 5, 9, 4, 8, 6, 8],

    'platform': ['Theatre', 'Theatre', 'OTT', 'Theatre', 'OTT',
                 'Theatre', 'OTT', 'Theatre', 'Theatre', 'Theatre',
                 'OTT', 'OTT', 'Theatre', 'OTT', 'Theatre',
                 'OTT', 'Theatre', 'OTT', 'OTT', 'OTT',
                 'Theatre', 'OTT', 'Theatre', 'Theatre', 'OTT',
                 'Theatre', 'OTT', 'Theatre', 'Theatre', 'Theatre',
                 'OTT', 'OTT', 'Theatre', 'OTT', 'Theatre',
                 'OTT', 'Theatre', 'OTT', 'OTT', 'OTT',
                 'Theatre', 'OTT', 'Theatre', 'Theatre', 'OTT',
                 'Theatre', 'OTT', 'Theatre', 'Theatre', 'Theatre'],

    'result': ['Hit', 'Hit', 'Flop', 'Hit', 'Flop',
               'Hit', 'Flop', 'Hit', 'Flop', 'Hit',
               'Flop', 'Flop', 'Hit', 'Flop', 'Hit',
               'Flop', 'Hit', 'Flop', 'Flop', 'Flop',
               'Hit', 'Flop', 'Hit', 'Hit', 'Flop',
               'Hit', 'Flop', 'Hit', 'Hit', 'Hit',
               'Flop', 'Flop', 'Hit', 'Flop', 'Hit',
               'Flop', 'Hit', 'Flop', 'Flop', 'Flop',
               'Hit', 'Flop', 'Hit', 'Hit', 'Flop',
               'Hit', 'Flop', 'Hit', 'Hit', 'Hit']
}

df = pd.DataFrame(data)

print("Dataset created!")
print(f"Total movies: {len(df)}")
print(f"\nHit/Flop distribution:")
print(df['result'].value_counts())
print("\nFirst 5 rows:")
print(df.head())
print("\n✅ Dataset ready!")

Dataset created!
Total movies: 50

Hit/Flop distribution:
result
Hit     26
Flop    24
Name: count, dtype: int64

First 5 rows:
   budget    genre  cast_popularity platform result
0     150   Action                9  Theatre    Hit
1     200   Comedy                7  Theatre    Hit
2      50   Horror                5      OTT   Flop
3     300   Action               10  Theatre    Hit
4      25  Romance                4      OTT   Flop

✅ Dataset ready!


In [3]:
# Cell 3 - Prepare data for training

# Encode categorical columns (convert text to numbers)
# ML models only understand numbers!

le_genre = LabelEncoder()
le_platform = LabelEncoder()
le_result = LabelEncoder()

df['genre_encoded'] = le_genre.fit_transform(df['genre'])
df['platform_encoded'] = le_platform.fit_transform(df['platform'])
df['result_encoded'] = le_result.fit_transform(df['result'])

print("Encoding mapping:")
print(f"Genres: {dict(zip(le_genre.classes_, le_genre.transform(le_genre.classes_)))}")
print(f"Platforms: {dict(zip(le_platform.classes_, le_platform.transform(le_platform.classes_)))}")
print(f"Results: {dict(zip(le_result.classes_, le_result.transform(le_result.classes_)))}")

# Select features for training
X = df[['budget', 'genre_encoded', 'cast_popularity', 'platform_encoded']]
y = df['result_encoded']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print("\n✅ Data prepared for training!")

Encoding mapping:
Genres: {'Action': np.int64(0), 'Comedy': np.int64(1), 'Drama': np.int64(2), 'Horror': np.int64(3), 'Romance': np.int64(4), 'Thriller': np.int64(5)}
Platforms: {'OTT': np.int64(0), 'Theatre': np.int64(1)}
Results: {'Flop': np.int64(0), 'Hit': np.int64(1)}

Training samples: 40
Testing samples: 10

✅ Data prepared for training!


In [4]:
# Cell 4 - Train 3 models and compare
# This covers your syllabus Ch 16-18 and 19-21!

print("="*50)
print("Training Decision Tree model...")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_predictions)
print(f"✅ Decision Tree Accuracy: {dt_accuracy * 100:.2f}%")

print("="*50)
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"✅ Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

print("="*50)
print("Training Logistic Regression model...")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f"✅ Logistic Regression Accuracy: {lr_accuracy * 100:.2f}%")

print("\n" + "="*50)
print("COMPARISON:")
print(f"Decision Tree:       {dt_accuracy * 100:.2f}%")
print(f"Random Forest:       {rf_accuracy * 100:.2f}%")
print(f"Logistic Regression: {lr_accuracy * 100:.2f}%")

# Select best model
best_accuracy = max(dt_accuracy, rf_accuracy, lr_accuracy)
if best_accuracy == rf_accuracy:
    best_model = rf_model
    print("\n🏆 Winner: Random Forest!")
elif best_accuracy == dt_accuracy:
    best_model = dt_model
    print("\n🏆 Winner: Decision Tree!")
else:
    best_model = lr_model
    print("\n🏆 Winner: Logistic Regression!")

print("\n✅ All models trained successfully!")

Training Decision Tree model...
✅ Decision Tree Accuracy: 100.00%
Training Random Forest model...
✅ Random Forest Accuracy: 100.00%
Training Logistic Regression model...
✅ Logistic Regression Accuracy: 90.00%

COMPARISON:
Decision Tree:       100.00%
Random Forest:       100.00%
Logistic Regression: 90.00%

🏆 Winner: Random Forest!

✅ All models trained successfully!


In [5]:
# Cell 5 - Evaluation with Confusion Matrix
# Covers your syllabus Ch 19-21!

print("=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, rf_predictions)
print(cm)

print("\n=== DETAILED REPORT ===")
print(classification_report(y_test, rf_predictions,
      target_names=le_result.classes_))

print("\n✅ Evaluation complete!")

=== CONFUSION MATRIX ===
[[6 0]
 [0 4]]

=== DETAILED REPORT ===
              precision    recall  f1-score   support

        Flop       1.00      1.00      1.00         6
         Hit       1.00      1.00      1.00         4

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


✅ Evaluation complete!


In [6]:
# Cell 6 - Save the model
import os

models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

# Save best model
model_path = os.path.join(models_dir, 'hitflop_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"✅ Model saved: {model_path}")

# Save label encoders
le_genre_path = os.path.join(models_dir, 'le_genre.pkl')
with open(le_genre_path, 'wb') as f:
    pickle.dump(le_genre, f)

le_platform_path = os.path.join(models_dir, 'le_platform.pkl')
with open(le_platform_path, 'wb') as f:
    pickle.dump(le_platform, f)

le_result_path = os.path.join(models_dir, 'le_result.pkl')
with open(le_result_path, 'wb') as f:
    pickle.dump(le_result, f)

print("✅ Label encoders saved!")

# Test saved model
print("\n=== TESTING SAVED MODEL ===")

test_movies = [
    {'title': 'King (SRK)', 'budget': 300, 'genre': 'Action', 'cast': 10, 'platform': 'Theatre'},
    {'title': 'Small Budget Film', 'budget': 15, 'genre': 'Drama', 'cast': 3, 'platform': 'OTT'},
    {'title': 'Pushpa 3', 'budget': 250, 'genre': 'Action', 'cast': 9, 'platform': 'Theatre'},
    {'title': 'Elvish Debut', 'budget': 25, 'genre': 'Comedy', 'cast': 5, 'platform': 'Theatre'},
]

print("\nPredictions:")
for movie in test_movies:
    genre_enc = le_genre.transform([movie['genre']])[0]
    platform_enc = le_platform.transform([movie['platform']])[0]
    features = [[movie['budget'], genre_enc, movie['cast'], platform_enc]]
    pred = best_model.predict(features)[0]
    prob = best_model.predict_proba(features)[0]
    result = le_result.inverse_transform([pred])[0]
    confidence = round(max(prob) * 100, 1)
    emoji = "🔥 HIT" if result == 'Hit' else "💀 FLOP"
    print(f"{movie['title']} → {emoji} ({confidence}% confident)")

print("\n✅ Hit/Flop model saved and tested!")
print("🎉 Your AI Hit/Flop Predictor is ready!")

✅ Model saved: D:\FilmFusion\Backend\models\hitflop_model.pkl
✅ Label encoders saved!

=== TESTING SAVED MODEL ===

Predictions:
King (SRK) → 🔥 HIT (100.0% confident)
Small Budget Film → 💀 FLOP (100.0% confident)
Pushpa 3 → 🔥 HIT (100.0% confident)
Elvish Debut → 💀 FLOP (73.0% confident)

✅ Hit/Flop model saved and tested!
🎉 Your AI Hit/Flop Predictor is ready!


D:\FilmFusion\Backend\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\FilmFusion\Backend\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\FilmFusion\Backend\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\FilmFusion\Backend\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
D:\FilmFusion\Backend\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  